# Machine Learning - DataCo Smart Supply Chain
En esta fase se aplican técnicas de Machine Learning al conjunto de datos con el objetivo de desarrollar modelos capaces de identificar situaciones de riesgo relevantes para la actividad empresarial. El análisis parte de los resultados obtenidos en la fase de exploración de datos, en la que se identificaron dos problemas especialmente relevantes: el riesgo de retraso en los envíos y la existencia de operaciones que generan pérdidas.

Para abordar estos problemas se desarrollarán dos modelos de clasificación independientes. El primero tendrá como objetivo predecir el riesgo de retraso de una operación, utilizando las características disponibles antes de conocer el resultado del envío. El segundo buscará predecir si una operación puede terminar generando pérdidas, utilizando como referencia el beneficio obtenido en la operación.

Antes de entrenar los modelos se realizará una selección de las variables disponibles, prestando especial atención a evitar problemas de fuga de información (data leakage). Posteriormente se prepararán los datos, se entrenarán los modelos y se evaluará su capacidad predictiva mediante diferentes métricas. Finalmente, se analizarán los resultados obtenidos para determinar qué variables aportan mayor información y qué utilidad pueden tener las predicciones desde un punto de vista empresarial.

Los modelos se desarrollarán de forma independiente del análisis exploratorio realizado anteriormente, utilizando el dataset limpio como punto de partida y manteniendo la estructura original del proyecto.


## 1. Preparación de los datos
## 2. Modelo 1: Predicción del riesgo de retraso
## 3. Modelo 2: Predicción de operaciones con pérdidas
## 4. Comparación y conclusiones

## 1. Preparación de los datos

En primer lugar, se carga el conjunto de datos limpio obtenido en la fase de análisis exploratorio. Se comprueba su estructura y dimensiones para verificar que el proceso de Machine Learning parte del mismo conjunto de datos utilizado en el análisis anterior.

A partir de este conjunto se prepararán posteriormente las variables necesarias para cada uno de los dos modelos, teniendo en cuenta las características que estarían disponibles antes de producirse el retraso o la pérdida.

In [1]:
# Importamos las librerías necesarias para la preparación, entrenamiento y evaluación de los modelos
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# Cargamos el dataset limpio y comprobamos sus dimensiones y primeras filas
df = pd.read_csv("../data/processed/DataCoSupplyChain_clean.csv")

print("Dimensiones del dataset:", df.shape)

display(df.head())

Dimensiones del dataset: (180519, 51)


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


In [3]:
# Mostramos todas las variables disponibles en el dataset limpio
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

1. type
2. days_for_shipping_real
3. days_for_shipment_scheduled
4. benefit_per_order
5. sales_per_customer
6. delivery_status
7. late_delivery_risk
8. category_id
9. category_name
10. customer_city
11. customer_country
12. customer_email
13. customer_fname
14. customer_id
15. customer_lname
16. customer_password
17. customer_segment
18. customer_state
19. customer_street
20. customer_zipcode
21. department_id
22. department_name
23. latitude
24. longitude
25. market
26. order_city
27. order_country
28. order_customer_id
29. order_date_dateorders
30. order_id
31. order_item_cardprod_id
32. order_item_discount
33. order_item_discount_rate
34. order_item_id
35. order_item_product_price
36. order_item_profit_ratio
37. order_item_quantity
38. sales
39. order_item_total
40. order_profit_per_order
41. order_region
42. order_state
43. order_status
44. product_card_id
45. product_category_id
46. product_image
47. product_name
48. product_price
49. product_status
50. shipping_date_dateorders
51

In [4]:
# Comprobamos si late_delivery_risk coincide con el estado real de entrega
pd.crosstab(df["late_delivery_risk"], df["delivery_status"])

delivery_status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
late_delivery_risk,,,,
0,41592,0,7754,32196
1,0,98977,0,0


## 2. Modelo 1: Predicción del riesgo de retraso

### Selección de variables

Para construir el modelo se seleccionarán únicamente aquellas variables que puedan aportar información útil para anticipar un posible retraso en la entrega.

Se excluirán, en primer lugar, las variables que contienen directamente el resultado del proceso de envío, como delivery_status, days_for_shipping_real y shipping_date_dateorders, ya que su utilización supondría disponer de información posterior al momento en el que se realiza la predicción. La variable late_delivery_risk tampoco se utilizará como predictor, puesto que constituye la variable objetivo del modelo.

También se excluirán los datos personales de los clientes, los identificadores de registros y otras variables que no tengan un significado predictivo directo. El objetivo es que el modelo utilice características relacionadas con el pedido, el producto, el cliente, la ubicación y las condiciones de envío que puedan estar disponibles antes de conocer el resultado de la entrega.

In [5]:
# Seleccionamos las variables disponibles antes del resultado del envío que utilizaremos para predecir el riesgo de retraso
columnas_predictoras_retraso = [
    "type",
    "days_for_shipment_scheduled",
    "category_name",
    "customer_city",
    "customer_country",
    "customer_state",
    "customer_segment",
    "department_name",
    "latitude",
    "longitude",
    "market",
    "order_city",
    "order_country",
    "order_date_dateorders",
    "order_item_discount",
    "order_item_discount_rate",
    "order_item_product_price",
    "order_item_quantity",
    "sales",
    "order_item_total",
    "order_region",
    "order_state",
    "product_name",
    "product_price",
    "product_status",
    "shipping_mode"
]

print("Número de variables predictoras:", len(columnas_predictoras_retraso))

Número de variables predictoras: 26


In [6]:
# Definimos como variable objetivo el riesgo de retraso de cada operación
variable_objetivo_retraso = "late_delivery_risk"

In [7]:
# Separamos las variables predictoras de la variable objetivo para preparar el modelo
X_retraso = df[columnas_predictoras_retraso].copy()
y_retraso = df[variable_objetivo_retraso].copy()

print("Dimensiones de X:", X_retraso.shape)
print("Dimensiones de y:", y_retraso.shape)

Dimensiones de X: (180519, 26)
Dimensiones de y: (180519,)


### División de los datos en entrenamiento y prueba

Una vez seleccionadas las variables predictoras y definida la variable objetivo, se divide el conjunto de datos en dos partes. El 80 % de los registros se utilizará para entrenar el modelo, mientras que el 20 % restante se reservará para evaluar posteriormente su capacidad de predicción sobre datos que no ha utilizado durante el entrenamiento.

La división se realiza de forma aleatoria y se utiliza la estratificación respecto a la variable late_delivery_risk, con el objetivo de mantener en los conjuntos de entrenamiento y prueba una proporción similar de operaciones con y sin retraso.

In [8]:
# Dividimos los datos en entrenamiento y prueba para evaluar el modelo con operaciones que no ha visto
X_train_retraso, X_test_retraso, y_train_retraso, y_test_retraso = train_test_split(
    X_retraso,
    y_retraso,
    test_size=0.2,
    random_state=42,
    stratify=y_retraso
)

print("Datos de entrenamiento:", X_train_retraso.shape)
print("Datos de prueba:", X_test_retraso.shape)

Datos de entrenamiento: (144415, 26)
Datos de prueba: (36104, 26)


In [9]:
# Identificamos las variables numéricas y categóricas que utilizaremos en el modelo
columnas_numericas_retraso = X_train_retraso.select_dtypes(include=["number"]).columns.tolist()
columnas_categoricas_retraso = X_train_retraso.select_dtypes(include=["object", "category"]).columns.tolist()

print("Variables numéricas:", len(columnas_numericas_retraso))
print(columnas_numericas_retraso)

print("\nVariables categóricas:", len(columnas_categoricas_retraso))
print(columnas_categoricas_retraso)

Variables numéricas: 11
['days_for_shipment_scheduled', 'latitude', 'longitude', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'sales', 'order_item_total', 'product_price', 'product_status']

Variables categóricas: 15
['type', 'category_name', 'customer_city', 'customer_country', 'customer_state', 'customer_segment', 'department_name', 'market', 'order_city', 'order_country', 'order_date_dateorders', 'order_region', 'order_state', 'product_name', 'shipping_mode']


C:\Users\acer\AppData\Local\Temp\ipykernel_13652\2276371188.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas_retraso = X_train_retraso.select_dtypes(include=["object", "category"]).columns.tolist()


In [10]:
# Creamos una copia del conjunto de entrenamiento para transformar la fecha sin modificar el dataset original
X_train_retraso = X_train_retraso.copy()

# Convertimos la fecha del pedido a formato fecha
X_train_retraso["order_date_dateorders"] = pd.to_datetime(
    X_train_retraso["order_date_dateorders"],
    errors="coerce"
)

# Extraemos el mes y el día de la semana como variables predictoras
X_train_retraso["order_month"] = X_train_retraso["order_date_dateorders"].dt.month
X_train_retraso["order_day_of_week"] = X_train_retraso["order_date_dateorders"].dt.dayofweek

# Eliminamos la fecha original porque el modelo no necesita cada fecha concreta
X_train_retraso = X_train_retraso.drop(columns=["order_date_dateorders"])

print("Dimensiones de X_train después de preparar la fecha:", X_train_retraso.shape)

Dimensiones de X_train después de preparar la fecha: (144415, 27)


In [11]:
# Preparamos el conjunto de prueba de la misma forma que el conjunto de entrenamiento
X_test_retraso = X_test_retraso.copy()

# Convertimos la fecha del pedido a formato fecha
X_test_retraso["order_date_dateorders"] = pd.to_datetime(
    X_test_retraso["order_date_dateorders"],
    errors="coerce"
)

# Extraemos el mes y el día de la semana como variables predictoras
X_test_retraso["order_month"] = X_test_retraso["order_date_dateorders"].dt.month
X_test_retraso["order_day_of_week"] = X_test_retraso["order_date_dateorders"].dt.dayofweek

# Eliminamos la fecha original
X_test_retraso = X_test_retraso.drop(columns=["order_date_dateorders"])

print("Dimensiones de X_test después de preparar la fecha:", X_test_retraso.shape)

Dimensiones de X_test después de preparar la fecha: (36104, 27)


In [12]:
# Actualizamos la clasificación de variables después de transformar la fecha del pedido
columnas_numericas_retraso = X_train_retraso.select_dtypes(include=["number"]).columns.tolist()
columnas_categoricas_retraso = X_train_retraso.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Variables numéricas:", len(columnas_numericas_retraso))
print(columnas_numericas_retraso)

print("\nVariables categóricas:", len(columnas_categoricas_retraso))
print(columnas_categoricas_retraso)

Variables numéricas: 13
['days_for_shipment_scheduled', 'latitude', 'longitude', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'sales', 'order_item_total', 'product_price', 'product_status', 'order_month', 'order_day_of_week']

Variables categóricas: 14
['type', 'category_name', 'customer_city', 'customer_country', 'customer_state', 'customer_segment', 'department_name', 'market', 'order_city', 'order_country', 'order_region', 'order_state', 'product_name', 'shipping_mode']


In [13]:
# Preparamos las variables numéricas y categóricas para que puedan ser utilizadas por el modelo
transformador_retraso = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="median")),
                ("escalado", StandardScaler())
            ]),
            columnas_numericas_retraso
        ),
        (
            "categoricas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="most_frequent")),
                ("codificacion", OneHotEncoder(handle_unknown="ignore"))
            ]),
            columnas_categoricas_retraso
        )
    ]
)

print("Transformador preparado correctamente.")

Transformador preparado correctamente.


### Preprocesamiento y modelo

Las variables seleccionadas presentan diferentes tipos de datos, por lo que es necesario realizar un preprocesamiento antes de entrenar el modelo. Las variables numéricas se estandarizan para que tengan una escala comparable, mientras que las variables categóricas se transforman mediante One-Hot Encoding.

Estas transformaciones se integran junto con una Regresión Logística en un único pipeline. De esta forma, el mismo proceso de preparación se aplica tanto a los datos de entrenamiento como a los datos de prueba, evitando realizar transformaciones diferentes entre ambos conjuntos.

In [14]:
# Creamos un pipeline que combina el preprocesamiento de los datos con el modelo de Regresión Logística
modelo_retraso = Pipeline([
    ("preprocesamiento", transformador_retraso),
    ("modelo", LogisticRegression(max_iter=1000))
])

print("Pipeline del modelo creado correctamente.")

Pipeline del modelo creado correctamente.


### Entrenamiento del modelo

Una vez definido el pipeline de preprocesamiento y el algoritmo de clasificación, se entrena el modelo utilizando exclusivamente el conjunto de datos de entrenamiento. Durante este proceso, el modelo aprende las relaciones entre las variables disponibles antes del envío y el riesgo de que la operación termine sufriendo un retraso.

El conjunto de prueba se mantiene separado y no interviene en este entrenamiento, ya que se utilizará posteriormente para evaluar el comportamiento del modelo sobre datos que no ha visto.

In [15]:
# Entrenamos el modelo de Regresión Logística utilizando únicamente los datos de entrenamiento
modelo_retraso.fit(X_train_retraso, y_train_retraso)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


### Predicción

In [16]:
# Generamos las predicciones del modelo sobre los datos de prueba que no utilizó durante el entrenamiento
y_pred_retraso = modelo_retraso.predict(X_test_retraso)

print("Número de predicciones:", len(y_pred_retraso))
print("Primeras 10 predicciones:", y_pred_retraso[:10])

Número de predicciones: 36104
Primeras 10 predicciones: [1 0 0 1 1 0 1 1 1 1]


### Evaluación de las predicciones

Las predicciones obtenidas se comparan con los valores reales de late_delivery_risk correspondientes al conjunto de prueba. De esta forma, podemos determinar en qué medida el modelo ha identificado correctamente las operaciones que presentan riesgo de retraso y aquellas que no lo presentan.

Para evaluar el modelo se utilizarán diferentes métricas de clasificación, ya que una única medida no permite conocer todos los aspectos de su rendimiento.

In [17]:
# Calculamos las principales métricas de clasificación para evaluar las predicciones del modelo
accuracy_retraso = accuracy_score(y_test_retraso, y_pred_retraso)
precision_retraso = precision_score(y_test_retraso, y_pred_retraso)
recall_retraso = recall_score(y_test_retraso, y_pred_retraso)
f1_retraso = f1_score(y_test_retraso, y_pred_retraso)

print(f"Accuracy: {accuracy_retraso:.4f}")
print(f"Precision: {precision_retraso:.4f}")
print(f"Recall: {recall_retraso:.4f}")
print(f"F1-score: {f1_retraso:.4f}")

Accuracy: 0.7135
Precision: 0.7839
Recall: 0.6592
F1-score: 0.7162


Los resultados obtenidos muestran que el modelo alcanza un Accuracy de 0,7135, lo que significa que clasifica correctamente aproximadamente el 71 % de las operaciones del conjunto de prueba.

La Precision es de 0,7839, por lo que, de todas las operaciones que el modelo identifica como posibles retrasos, aproximadamente el 78 % terminan siendo realmente operaciones con retraso. Este resultado indica que el modelo tiene una capacidad razonable para evitar falsas alarmas.

El Recall es de 0,6592, lo que significa que el modelo consigue identificar aproximadamente el 66 % de las operaciones que realmente sufren un retraso. Por tanto, existe una parte importante de los retrasos reales que el modelo no consigue detectar.

Por último, el F1-score es de 0,7162, lo que refleja un equilibrio entre la precisión y la capacidad del modelo para detectar los retrasos reales.

En conjunto, el modelo presenta un comportamiento moderado. Su precisión al identificar posibles retrasos es relativamente buena, pero su capacidad para detectar todos los casos reales de retraso es más limitada. Por tanto, puede resultar útil como herramienta de apoyo para identificar operaciones con mayor riesgo, aunque sus predicciones no deberían considerarse suficientes por sí solas para anticipar todos los retrasos.

### Matriz de confusión
Nos permitirá ver exactamente cuántos retrasos detectamos correctamente y cuántos se nos escapan.

In [18]:
# Calculamos la matriz de confusión para conocer los aciertos y errores del modelo
matriz_retraso = confusion_matrix(y_test_retraso, y_pred_retraso)

print("Matriz de confusión:")
print(matriz_retraso)

Matriz de confusión:
[[12711  3597]
 [ 6746 13050]]


La matriz de confusión permite analizar con mayor detalle los aciertos y errores cometidos por el modelo.

Se obtienen 12.711 verdaderos negativos, correspondientes a operaciones que no sufrieron retraso y que el modelo clasificó correctamente como operaciones sin riesgo de retraso.

También se identifican 13.050 verdaderos positivos, correspondientes a operaciones que realmente sufrieron un retraso y que el modelo consiguió identificar correctamente.

Por otro lado, aparecen 3.597 falsos positivos. En estos casos, el modelo predijo que una operación sufriría un retraso, pero finalmente no se produjo. Estos casos representan falsas alarmas para la empresa.

El resultado más relevante son los 6.746 falsos negativos. Se trata de operaciones que finalmente sufrieron un retraso, pero que el modelo había clasificado como operaciones sin riesgo. Desde un punto de vista empresarial, estos errores son especialmente importantes, ya que representan situaciones de retraso que el modelo no consiguió anticipar.

Por tanto, aunque el modelo identifica correctamente una parte importante de las operaciones con retraso, todavía existe un número considerable de retrasos reales que pasan desapercibidos. Este resultado es coherente con el Recall obtenido anteriormente y será relevante a la hora de valorar la utilidad del modelo para la toma de decisiones.

In [19]:
# Calculamos la capacidad del modelo para distinguir entre operaciones con y sin retraso
y_prob_retraso = modelo_retraso.predict_proba(X_test_retraso)[:, 1]

roc_auc_retraso = roc_auc_score(y_test_retraso, y_prob_retraso)

print(f"ROC-AUC: {roc_auc_retraso:.4f}")

ROC-AUC: 0.7812


El modelo obtiene un ROC-AUC de 0,7812, lo que indica una capacidad razonable para distinguir entre las operaciones que terminan sufriendo un retraso y aquellas que no lo hacen.

El valor obtenido se encuentra claramente por encima de 0,5, que representaría un comportamiento equivalente a una clasificación aleatoria, aunque todavía está lejos de 1, que correspondería a una separación perfecta entre ambas clases.

Este resultado confirma que las variables seleccionadas contienen información útil para diferenciar las operaciones con mayor y menor riesgo de retraso. Sin embargo, el modelo todavía presenta limitaciones, especialmente en la detección de una parte de los retrasos reales, como ya se observó en el Recall y en la matriz de confusión.

### Optimización del modelo

Aunque el modelo inicial presenta una capacidad razonable para identificar operaciones con riesgo de retraso, los resultados muestran que todavía existe margen de mejora, especialmente en la detección de los retrasos reales.

Para intentar mejorar su rendimiento se utilizará GridSearchCV, una técnica que permite probar diferentes combinaciones de parámetros del modelo mediante validación cruzada. En este caso se buscará la configuración que consiga un mejor Recall, ya que el objetivo principal es reducir el número de operaciones con retraso que el modelo no consigue detectar.

El modelo obtenido mediante este proceso se comparará posteriormente con el modelo inicial para comprobar si la optimización consigue mejorar su capacidad de detección.

In [20]:
# Buscamos la mejor configuración de la Regresión Logística utilizando validación cruzada y priorizando el Recall
parametros_retraso = {
    "modelo__C": [0.01, 0.1, 1, 10, 100]
}

grid_retraso = GridSearchCV(
    modelo_retraso,
    parametros_retraso,
    scoring="recall",
    cv=5,
    n_jobs=-1
)

grid_retraso.fit(X_train_retraso, y_train_retraso)

print("Mejores parámetros:", grid_retraso.best_params_)
print("Mejor Recall en validación cruzada:", grid_retraso.best_score_)

Mejores parámetros: {'modelo__C': 100}
Mejor Recall en validación cruzada: 0.6623937364043198


In [21]:
# Generamos las predicciones del modelo optimizado sobre el conjunto de prueba
y_pred_retraso_optimizado = grid_retraso.predict(X_test_retraso)

print("Número de predicciones:", len(y_pred_retraso_optimizado))

Número de predicciones: 36104


In [22]:
# Calculamos las métricas del modelo optimizado para compararlas con el modelo inicial
accuracy_retraso_opt = accuracy_score(y_test_retraso, y_pred_retraso_optimizado)
precision_retraso_opt = precision_score(y_test_retraso, y_pred_retraso_optimizado)
recall_retraso_opt = recall_score(y_test_retraso, y_pred_retraso_optimizado)
f1_retraso_opt = f1_score(y_test_retraso, y_pred_retraso_optimizado)

print(f"Accuracy: {accuracy_retraso_opt:.4f}")
print(f"Precision: {precision_retraso_opt:.4f}")
print(f"Recall: {recall_retraso_opt:.4f}")
print(f"F1-score: {f1_retraso_opt:.4f}")

Accuracy: 0.7126
Precision: 0.7795
Recall: 0.6636
F1-score: 0.7169


In [23]:
# Comparamos distintos umbrales de clasificación para intentar detectar un mayor número de retrasos reales
y_prob_retraso_optimizado = grid_retraso.predict_proba(X_test_retraso)[:, 1]

umbrales_retraso = [0.9, 0.8, 0.7, 0.60, 0.50, 0.45, 0.40, 0.35, 0.30, 0.20, 0.10]

for umbral in umbrales_retraso:
    y_pred_umbral = (y_prob_retraso_optimizado >= umbral).astype(int)

    precision = precision_score(y_test_retraso, y_pred_umbral)
    recall = recall_score(y_test_retraso, y_pred_umbral)
    f1 = f1_score(y_test_retraso, y_pred_umbral)

    print(f"\nUmbral: {umbral:.2f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")


Umbral: 0.90
Precision: 0.9491
Recall: 0.2974
F1-score: 0.4529

Umbral: 0.80
Precision: 0.9032
Recall: 0.4206
F1-score: 0.5739

Umbral: 0.70
Precision: 0.8664
Recall: 0.5282
F1-score: 0.6563

Umbral: 0.60
Precision: 0.8324
Recall: 0.5916
F1-score: 0.6916

Umbral: 0.50
Precision: 0.7795
Recall: 0.6636
F1-score: 0.7169

Umbral: 0.45
Precision: 0.7386
Recall: 0.7158
F1-score: 0.7270

Umbral: 0.40
Precision: 0.6926
Recall: 0.7832
F1-score: 0.7351

Umbral: 0.35
Precision: 0.6445
Recall: 0.8539
F1-score: 0.7346

Umbral: 0.30
Precision: 0.6125
Recall: 0.9085
F1-score: 0.7317

Umbral: 0.20
Precision: 0.5740
Recall: 0.9780
F1-score: 0.7234

Umbral: 0.10
Precision: 0.5591
Recall: 0.9974
F1-score: 0.7165


In [24]:
# Calculamos la matriz de confusión utilizando el umbral de 0,40 seleccionado para priorizar la detección de retrasos
umbral_optimo_retraso = 0.40

y_pred_retraso_umbral = (
    y_prob_retraso_optimizado >= umbral_optimo_retraso
).astype(int)

matriz_retraso_umbral = confusion_matrix(
    y_test_retraso,
    y_pred_retraso_umbral
)

print("Matriz de confusión con umbral 0,40:")
print(matriz_retraso_umbral)

Matriz de confusión con umbral 0,40:
[[ 9426  6882]
 [ 4292 15504]]


La matriz de confusión muestra los resultados obtenidos por el modelo final utilizando un umbral de clasificación de 0,40.

El modelo identifica correctamente 9.426 operaciones sin retraso y 15.504 operaciones que realmente sufren un retraso. Por otro lado, clasifica erróneamente 6.882 operaciones como retrasadas cuando finalmente no lo están, mientras que 4.292 operaciones que sí sufren un retraso no son detectadas por el modelo.

En total, el modelo acierta en 24.930 de las 36.104 operaciones analizadas (69,05 %). El resultado refleja la decisión de utilizar un umbral más bajo para detectar un mayor número de retrasos reales, aunque esto implique generar más falsas alarmas.

In [25]:
# Calculamos el ROC-AUC del modelo optimizado para evaluar su capacidad de distinguir entre operaciones con y sin retraso
roc_auc_retraso_optimizado = roc_auc_score(
    y_test_retraso,
    y_prob_retraso_optimizado
)

print(f"ROC-AUC del modelo optimizado: {roc_auc_retraso_optimizado:.4f}")

ROC-AUC del modelo optimizado: 0.7824


### Conclusión del Modelo 1: Predicción del riesgo de retraso

El modelo inicial de Regresión Logística obtuvo un Recall de 0,6592 y un ROC-AUC de 0,7812. Posteriormente, mediante GridSearchCV se optimizó su configuración, seleccionando C = 100, con una ligera mejora del Recall hasta 0,6636 y del ROC-AUC hasta 0,7824.

A continuación, se analizaron diferentes umbrales de clasificación. Se seleccionó un umbral de 0,40, ya que obtuvo el mejor F1-score (0,7351) y permitió aumentar el Recall hasta 0,7832. Esta mejora implica reducir la Precision hasta 0,6926 y aumentar considerablemente las falsas alarmas, pero permite detectar más retrasos reales.

Por tanto, el modelo final será la Regresión Logística optimizada mediante GridSearchCV, con C = 100 y un umbral de 0,40, priorizando la detección de retrasos frente a una mayor precisión en las alertas.

 ## Preparación de las predicciones del modelo de retrasos para Power BI

Una vez seleccionado y evaluado el modelo final de predicción del riesgo de retraso, se generan predicciones para el conjunto completo de operaciones con el objetivo de integrar sus resultados posteriormente en la base de datos y utilizarlos en Power BI.

Para esta fase se mantiene la configuración seleccionada durante la evaluación del modelo: Regresión Logística optimizada mediante GridSearchCV y un umbral de clasificación de 0,40.

Además de la clasificación binaria del riesgo, se conserva la probabilidad estimada de retraso para cada operación, lo que permitirá realizar análisis de riesgo más detallados en el dashboard.

In [26]:
# Preparar el conjunto completo de operaciones para generar las predicciones
# que posteriormente se utilizarán en la base de datos y en Power BI.

# Partimos de las mismas variables predictoras utilizadas durante el entrenamiento.
X_retraso_completo = df[columnas_predictoras_retraso].copy()

# Convertimos la fecha del pedido al mismo formato utilizado durante
# la preparación de los datos de entrenamiento y prueba.
X_retraso_completo["order_date_dateorders"] = pd.to_datetime(
    X_retraso_completo["order_date_dateorders"],
    errors="coerce"
)

# Extraemos el mes y el día de la semana como variables predictoras,
# utilizando exactamente la misma transformación aplicada al modelo.
X_retraso_completo["order_month"] = (
    X_retraso_completo["order_date_dateorders"].dt.month
)

X_retraso_completo["order_day_of_week"] = (
    X_retraso_completo["order_date_dateorders"].dt.dayofweek
)

# Eliminamos la fecha original porque el modelo utiliza las variables
# derivadas de ella en lugar de la fecha completa.
X_retraso_completo = X_retraso_completo.drop(
    columns=["order_date_dateorders"]
)

print("Dimensiones del conjunto completo:", X_retraso_completo.shape)

Dimensiones del conjunto completo: (180519, 27)


In [27]:
# Entrenamos una versión final del modelo optimizado utilizando
# todas las operaciones disponibles.
#
# Esto no modifica las métricas obtenidas anteriormente, ya que aquellas
# proceden exclusivamente del conjunto de prueba reservado para evaluación.

modelo_retraso_final = grid_retraso.best_estimator_

modelo_retraso_final.fit(
    X_retraso_completo,
    y_retraso
)

print("Modelo final entrenado correctamente con todas las operaciones.")

Modelo final entrenado correctamente con todas las operaciones.


In [28]:
# Generamos la probabilidad estimada de retraso para todas las operaciones.
probabilidad_retraso_completa = (
    modelo_retraso_final.predict_proba(X_retraso_completo)[:, 1]
)

# Convertimos las probabilidades en una clasificación binaria
# utilizando el umbral de 0,40 seleccionado durante la evaluación.
prediccion_retraso_completa = (
    probabilidad_retraso_completa >= umbral_optimo_retraso
).astype(int)

print(
    "Número de probabilidades generadas:",
    len(probabilidad_retraso_completa)
)

print(
    "Número de predicciones generadas:",
    len(prediccion_retraso_completa)
)

Número de probabilidades generadas: 180519
Número de predicciones generadas: 180519


In [29]:
# Creamos una tabla específica con los resultados del modelo.
# Esta tabla será la que posteriormente incorporaremos a la base de datos.

ml_delay_predictions = pd.DataFrame({
    "order_item_id": df["order_item_id"].values,
    "predicted_delay": prediccion_retraso_completa,
    "delay_probability": probabilidad_retraso_completa,
    "actual_late_delivery_risk": df["late_delivery_risk"].values
})

# Comprobamos que cada operación aparece una sola vez.
print(
    "Número de operaciones:",
    len(ml_delay_predictions)
)

print(
    "Identificadores únicos:",
    ml_delay_predictions["order_item_id"].nunique()
)

# Mostramos las primeras filas de la tabla.
ml_delay_predictions.head(10)

Número de operaciones: 180519
Identificadores únicos: 180519


,order_item_id,predicted_delay,delay_probability,actual_late_delivery_risk
0,180517,1,0.413349,0
1,179254,0,0.352419,1
2,179253,1,0.443749,0
3,179252,0,0.243912,0
4,179251,0,0.241232,0
5,179250,0,0.328262,0
6,179249,1,0.965553,1
7,179248,1,0.944043,1
8,179247,1,0.784546,1
9,179246,1,0.978043,1


In [30]:
# Comprobamos la distribución de las predicciones generadas.

print("Distribución de las predicciones:")
print(
    ml_delay_predictions["predicted_delay"]
    .value_counts()
    .sort_index()
)

print("\nProbabilidad media de retraso:")
print(
    ml_delay_predictions["delay_probability"].mean()
)

print("\nPrimeras filas de la tabla final:")
display(ml_delay_predictions.head(10))

Distribución de las predicciones:
predicted_delay
0     67922
1    112597
Name: count, dtype: int64

Probabilidad media de retraso:
0.548287216452043

Primeras filas de la tabla final:


,order_item_id,predicted_delay,delay_probability,actual_late_delivery_risk
0,180517,1,0.413349,0
1,179254,0,0.352419,1
2,179253,1,0.443749,0
3,179252,0,0.243912,0
4,179251,0,0.241232,0
5,179250,0,0.328262,0
6,179249,1,0.965553,1
7,179248,1,0.944043,1
8,179247,1,0.784546,1
9,179246,1,0.978043,1


In [31]:
# Guarda el DataFrame de predicciones en un CSV en la carpeta del proyecto
ml_delay_predictions.to_csv("ml_delay_predictions.csv", index=False)

print("¡CSV de predicciones guardado correctamente!")

¡CSV de predicciones guardado correctamente!


## 3. Modelo 2: Predicción de operaciones con pérdidas
En este segundo modelo se analizará la posibilidad de predecir si una operación terminará generando pérdidas. Durante el análisis exploratorio se identificó que aproximadamente el 18 % de las operaciones presentan un beneficio negativo, por lo que resulta de interés desarrollar un modelo capaz de identificar este tipo de operaciones antes de que se produzca su resultado final.

Al igual que en el modelo anterior, se utilizarán únicamente variables que puedan estar disponibles antes de conocer el beneficio definitivo de la operación. De esta forma, se evitará utilizar información que permita conocer directamente si la operación ha generado pérdidas y se simulará un escenario en el que el modelo pueda aplicarse posteriormente a nuevas transacciones.

In [32]:
# Creamos la variable objetivo: 1 si la operación genera pérdidas y 0 si no
df["operacion_con_perdida"] = (df["benefit_per_order"] < 0).astype(int)

print("Distribución de operaciones con y sin pérdidas:")
print(df["operacion_con_perdida"].value_counts())

print("\nPorcentaje de operaciones con pérdidas:")
print(df["operacion_con_perdida"].mean())

Distribución de operaciones con y sin pérdidas:
operacion_con_perdida
0    146735
1     33784
Name: count, dtype: int64

Porcentaje de operaciones con pérdidas:
0.18714927514555255


### Selección de variables

Para la construcción del modelo de predicción de pérdidas es fundamental seleccionar únicamente aquellas variables que estén disponibles en el momento previo a la ejecución o cierre definitivo de la operación.

Se excluyen todas las variables relacionadas con el resultado económico final de la transacción y al proceso posterior de envío e identificadores que no aportan capacidad explicativa.

Las variables predictoras seleccionadas abarcan información sobre las condiciones comerciales, importes, descuentos, productos, clientes y aspectos geográficos disponibles al registrar el pedido.

In [33]:
# Seleccionamos las variables disponibles antes de conocer el beneficio definitivo
columnas_predictoras_perdida = [
    "type",
    "days_for_shipment_scheduled",
    "sales_per_customer",
    "category_name",
    "customer_city",
    "customer_country",
    "customer_state",
    "customer_segment",
    "department_name",
    "latitude",
    "longitude",
    "market",
    "order_city",
    "order_country",
    "order_date_dateorders",
    "order_item_discount",
    "order_item_discount_rate",
    "order_item_product_price",
    "order_item_quantity",
    "sales",
    "order_item_total",
    "order_region",
    "order_state",
    "product_name",
    "product_price",
    "product_status",
    "shipping_mode"
]

variable_objetivo_perdida = "operacion_con_perdida"

# Separamos las variables predictoras y la variable objetivo
X_perdida = df[columnas_predictoras_perdida].copy()
y_perdida = df[variable_objetivo_perdida].copy()

print("Dimensiones de X:", X_perdida.shape)
print("Dimensiones de y:", y_perdida.shape)

Dimensiones de X: (180519, 27)
Dimensiones de y: (180519,)


### División en Train/Test y extracción de fechas

In [34]:
# Train / Test split con estratificación debido al desbalance del target (~18.7%)
X_train_perdida, X_test_perdida, y_train_perdida, y_test_perdida = train_test_split(
    X_perdida,
    y_perdida,
    test_size=0.2,
    random_state=42,
    stratify=y_perdida
)

# Procesamiento de la fecha
for df_split in [X_train_perdida, X_test_perdida]:
    df_split["order_date_dateorders"] = pd.to_datetime(df_split["order_date_dateorders"], errors="coerce")
    df_split["order_month"] = df_split["order_date_dateorders"].dt.month
    df_split["order_day_of_week"] = df_split["order_date_dateorders"].dt.dayofweek
    df_split.drop(columns=["order_date_dateorders"], inplace=True)

print("Datos de entrenamiento:", X_train_perdida.shape)
print("Datos de prueba:", X_test_perdida.shape)

Datos de entrenamiento: (144415, 28)
Datos de prueba: (36104, 28)


### Preprocesamiento y modelo base (Regresión Logística)

Las variables seleccionadas presentan distintos tipos de datos, por lo que se define un *Pipeline* de preprocesamiento similar al del Modelo 1:
* **Variables numéricas:** Imputación de valores faltantes por la mediana y estandarización.
* **Variables categóricas:** Imputación por el valor más frecuente y codificación mediante *One-Hot Encoding*.

En esta primera aproximación se utiliza una **Regresión Logística** con el parámetro `class_weight='balanced'`. Esto permite penalizar en mayor medida los errores en la clase minoritaria (operaciones con pérdidas, ~18,7 %) y establece un modelo base (*baseline*) para comparar posteriormente con algoritmos más complejos.

### Entrenamiento del Modelo Base

In [35]:
# Clasificamos las columnas según su tipo tras procesar la fecha
columnas_numericas_perdida = X_train_perdida.select_dtypes(include=["number"]).columns.tolist()
columnas_categoricas_perdida = X_train_perdida.select_dtypes(include=["object", "category", "string"]).columns.tolist()

# Preprocesador de columnas
transformador_perdida = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="median")),
                ("escalado", StandardScaler())
            ]),
            columnas_numericas_perdida
        ),
        (
            "categoricas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="most_frequent")),
                ("codificacion", OneHotEncoder(handle_unknown="ignore"))
            ]),
            columnas_categoricas_perdida
        )
    ]
)

# Pipeline del modelo base (Regresión Logística Balanceada)
modelo_lr_perdida = Pipeline([
    ("preprocesamiento", transformador_perdida),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Entrenamos el modelo base
modelo_lr_perdida.fit(X_train_perdida, y_train_perdida)

print("Modelo base de Regresión Logística entrenado correctamente.")

Modelo base de Regresión Logística entrenado correctamente.


### Predicción y evaluación del modelo base

Evaluamos el comportamiento de la Regresión Logística sobre el conjunto de test reservado mediante las principales métricas de clasificación y la matriz de confusión.

In [36]:
# Predicciones y probabilidades
y_pred_lr_perdida = modelo_lr_perdida.predict(X_test_perdida)
y_prob_lr_perdida = modelo_lr_perdida.predict_proba(X_test_perdida)[:, 1]

# Métricas
accuracy_lr = accuracy_score(y_test_perdida, y_pred_lr_perdida)
precision_lr = precision_score(y_test_perdida, y_pred_lr_perdida)
recall_lr = recall_score(y_test_perdida, y_pred_lr_perdida)
f1_lr = f1_score(y_test_perdida, y_pred_lr_perdida)
roc_auc_lr = roc_auc_score(y_test_perdida, y_prob_lr_perdida)

print(f"Accuracy (Regresión Logística): {accuracy_lr:.4f}")
print(f"Precision (Regresión Logística): {precision_lr:.4f}")
print(f"Recall (Regresión Logística): {recall_lr:.4f}")
print(f"F1-score (Regresión Logística): {f1_lr:.4f}")
print(f"ROC-AUC (Regresión Logística): {roc_auc_lr:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test_perdida, y_pred_lr_perdida))

Accuracy (Regresión Logística): 0.5261
Precision (Regresión Logística): 0.2006
Recall (Regresión Logística): 0.5134
F1-score (Regresión Logística): 0.2885
ROC-AUC (Regresión Logística): 0.5257

Matriz de confusión:
[[15526 13821]
 [ 3288  3469]]


### Análisis de los resultados del modelo base (Regresión Logística)

El análisis de las métricas obtenidas por el modelo base de Regresión Logística muestra un rendimiento insatisfactorio para la predicción de operaciones con pérdidas:

* **Baja capacidad de discriminación (ROC-AUC = 0,5257):** Un valor de ROC-AUC tan cercano a 0,5 indica que el comportamiento del modelo es prácticamente equivalente al de una clasificación aleatoria (lanzar una moneda al aire). El modelo lineal no está logrando identificar los patrones que separan las operaciones rentables de las que generan pérdidas.
* **Elevada tasa de falsas alarmas (Precision = 0,2006):** De todas las operaciones que el modelo marca como posibles pérdidas, solo el 20 % son realmente pérdidas. Esto genera 13.821 falsos positivos en el conjunto de test, lo que en un entorno operativo real supondría bloquear o revisar innecesariamente una gran cantidad de pedidos rentables.
* **Detección limitada (Recall = 0,5134):** El modelo solo logra detectar el 51,3 % de las pérdidas reales (3.469 de 6.757 casos), dejando pasar 3.288 operaciones con déficit financiero sin identificar.

**Conclusión intermedia:** 
Los resultados confirman que la relación entre las variables comerciales y la generación de pérdidas es **altamente no lineal y compleja**. Una Regresión Logística basada en combinaciones lineales de variables no es capaz de capturar el impacto conjunto de los descuentos, precios y categorías. 

Este comportamiento justifica plenamente dar el paso hacia un modelo avanzado basado en árboles de decisión como **XGBoost**, capaz de encontrar reglas de decisión complejas e interacciones no lineales entre variables.

### Modelo avanzado: XGBoost Classifier

Para capturar la complejidad no lineal entre precios, descuentos, productos y ubicaciones geograficas, se implementa un algoritmo basado en *Gradient Boosted Decision Trees* utilizando la librería **XGBoost**.

Con el fin de mitigar el desbalance de la clase minoritaria (18,7 % de operaciones con pérdidas), se calcula la proporción entre observaciones negativas y positivas para configurar el hiperparámetro `scale_pos_weight`, garantizando que el algoritmo asigne mayor peso al error en la detección de pérdidas.

In [37]:
# Calculamos la proporción para el balanceo de clases en XGBoost
ratio_desbalance = (y_train_perdida == 0).sum() / (y_train_perdida == 1).sum()

# Creamos el pipeline integrando el preprocesamiento con XGBoost
modelo_xgb_perdida = Pipeline([
    ("preprocesamiento", transformador_perdida),
    ("modelo", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=ratio_desbalance,
        random_state=42,
        n_jobs=-1
    ))
])

# Entrenamos el modelo avanzado
modelo_xgb_perdida.fit(X_train_perdida, y_train_perdida)

print("Modelo XGBoost entrenado correctamente.")

Modelo XGBoost entrenado correctamente.


In [38]:
# Predicciones de clase y probabilidades en el conjunto de prueba
y_pred_xgb = modelo_xgb_perdida.predict(X_test_perdida)
y_proba_xgb = modelo_xgb_perdida.predict_proba(X_test_perdida)[:, 1]

# Cálculo de métricas
acc_xgb = accuracy_score(y_test_perdida, y_pred_xgb)
prec_xgb = precision_score(y_test_perdida, y_pred_xgb)
rec_xgb = recall_score(y_test_perdida, y_pred_xgb)
f1_xgb = f1_score(y_test_perdida, y_pred_xgb)
roc_xgb = roc_auc_score(y_test_perdida, y_proba_xgb)

print("=== EVALUACIÓN MODELO XGBOOST ===")
print(f"Accuracy:  {acc_xgb:.4f}")
print(f"Precision: {prec_xgb:.4f}")
print(f"Recall:    {rec_xgb:.4f}")
print(f"F1-Score:  {f1_xgb:.4f}")
print(f"ROC-AUC:   {roc_xgb:.4f}\n")

print("--- Matriz de Confusión ---")
print(confusion_matrix(y_test_perdida, y_pred_xgb))

print("\n--- Informe de Clasificación ---")
print(classification_report(y_test_perdida, y_pred_xgb))

=== EVALUACIÓN MODELO XGBOOST ===
Accuracy:  0.4442
Precision: 0.1931
Recall:    0.6198
F1-Score:  0.2945
ROC-AUC:   0.5161

--- Matriz de Confusión ---
[[11850 17497]
 [ 2569  4188]]

--- Informe de Clasificación ---
              precision    recall  f1-score   support

           0       0.82      0.40      0.54     29347
           1       0.19      0.62      0.29      6757

    accuracy                           0.44     36104
   macro avg       0.51      0.51      0.42     36104
weighted avg       0.70      0.44      0.50     36104



In [39]:
# Inspección de las variables que entraron al entrenamiento
print(f"Número total de variables en X_train: {X_train_perdida.shape[1]}")
print("\n--- Lista de columnas utilizadas ---")
print(X_train_perdida.columns.tolist())

print("\n--- Tipos de datos por columna ---")
print(X_train_perdida.dtypes)

Número total de variables en X_train: 28

--- Lista de columnas utilizadas ---
['type', 'days_for_shipment_scheduled', 'sales_per_customer', 'category_name', 'customer_city', 'customer_country', 'customer_state', 'customer_segment', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'sales', 'order_item_total', 'order_region', 'order_state', 'product_name', 'product_price', 'product_status', 'shipping_mode', 'order_month', 'order_day_of_week']

--- Tipos de datos por columna ---
type                               str
days_for_shipment_scheduled      int64
sales_per_customer             float64
category_name                      str
customer_city                      str
customer_country                   str
customer_state                     str
customer_segment                   str
department_name                    str
latitude                     

In [40]:
# 1. Creamos la variable sintética de retraso en el envío si existen los días reales y programados
if 'days_for_shipping_real' in df.columns and 'days_for_shipment_scheduled' in df.columns:
    df['shipping_delay'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']

# 2. Definimos las columnas candidatas a usar (buscando solo en df.columns)
candidatas = [
    # Métricas financieras y de orden
    'sales_per_customer', 'order_item_discount', 'order_item_discount_rate',
    'order_item_product_price', 'order_item_quantity', 'sales', 
    'order_item_total', 'product_price',
    
    # Logística y tiempos
    'days_for_shipment_scheduled', 'days_for_shipping_real', 'shipping_delay',
    'late_delivery_risk', 'order_status',
    
    # Categóricas de baja/media cardinalidad (útiles para patrones)
    'type', 'customer_segment', 'market', 'order_region', 
    'shipping_mode', 'department_name'
]

# Seleccionamos ÚNICAMENTE las columnas que sí existen dentro de tu DataFrame `df`
cols_utiles = [col for col in candidatas if col in df.columns]

# 3. Re-creamos el conjunto de características
X_perdida_v2 = df[cols_utiles].copy()
y_perdida_v2 = df['operacion_con_perdida']

# 4. Nueva división Train / Test
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_perdida_v2, y_perdida_v2, test_size=0.20, random_state=42, stratify=y_perdida_v2
)

# 5. Definimos tipos de columnas para preprocesamiento
num_cols = X_train_v2.select_dtypes(include=['int64', 'int32', 'float64']).columns.tolist()
cat_cols = X_train_v2.select_dtypes(include=['object', 'string']).columns.tolist()

preprocesador_v2 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

print(" Columnas seleccionadas con éxito sin error de índice.")
print(f"Número total de features a entrenar: {len(cols_utiles)}")

 Columnas seleccionadas con éxito sin error de índice.
Número total de features a entrenar: 19


In [41]:
ratio_desbalance = (y_train_v2 == 0).sum() / (y_train_v2 == 1).sum()

modelo_xgb_v2 = Pipeline([
    ("preprocesamiento", preprocesador_v2),
    ("modelo", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=ratio_desbalance,
        random_state=42,
        n_jobs=-1
    ))
])

modelo_xgb_v2.fit(X_train_v2, y_train_v2)

# Predicciones
y_pred_v2 = modelo_xgb_v2.predict(X_test_v2)
y_proba_v2 = modelo_xgb_v2.predict_proba(X_test_v2)[:, 1]

print("=== NUEVA EVALUACIÓN XGBOOST (V2) ===")
print(f"ROC-AUC:   {roc_auc_score(y_test_v2, y_proba_v2):.4f}")
print(f"Accuracy:  {accuracy_score(y_test_v2, y_pred_v2):.4f}")
print(f"Recall:    {recall_score(y_test_v2, y_pred_v2):.4f}")
print(f"Precision: {precision_score(y_test_v2, y_pred_v2):.4f}")

=== NUEVA EVALUACIÓN XGBOOST (V2) ===
ROC-AUC:   0.5061
Accuracy:  0.5214
Recall:    0.4720
Precision: 0.1887


In [42]:
# Analizamos la tasa de pérdidas según el riesgo de entrega tardía
if 'late_delivery_risk' in df.columns:
    tasa_perdida_por_retraso = df.groupby('late_delivery_risk')['operacion_con_perdida'].mean() * 100
    print("=== TASA DE PÉRDIDA SEGÚN RETRASO EN EL ENVÍO ===")
    print(f"Pedidos A TIEMPO (0):  {tasa_perdida_por_retraso.get(0, 0):.2f}% de operaciones con pérdida")
    print(f"Pedidos CON RETRASO (1): {tasa_perdida_por_retraso.get(1, 0):.2f}% de operaciones con pérdida")

=== TASA DE PÉRDIDA SEGÚN RETRASO EN EL ENVÍO ===
Pedidos A TIEMPO (0):  18.68% de operaciones con pérdida
Pedidos CON RETRASO (1): 18.74% de operaciones con pérdida


In [43]:

# Comprobación de la relación entre días de retraso y el beneficio medio
if 'days_for_shipping_real' in df.columns and 'days_for_shipment_scheduled' in df.columns:
    df['dias_retraso_reales'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
    
    analisis_dias = df.groupby('dias_retraso_reales').agg(
        total_pedidos=('benefit_per_order', 'count'),
        tasa_perdida=('operacion_con_perdida', 'mean'),
        beneficio_promedio=('benefit_per_order', 'mean')
    ).reset_index()
    
    analisis_dias['tasa_perdida'] = analisis_dias['tasa_perdida'] * 100
    print("=== ANÁLISIS DE PÉRDIDA POR DÍAS EXACTOS DE RETRASO ===")
    print(analisis_dias.to_string(index=False))

=== ANÁLISIS DE PÉRDIDA POR DÍAS EXACTOS DE RETRASO ===
 dias_retraso_reales  total_pedidos  tasa_perdida  beneficio_promedio
                  -2          21666     18.332872           23.423528
                  -1          21700     19.000000           21.516997
                   0          33753     18.570201           22.532021
                   1          60647     18.797302           21.940582
                   2          28718     18.967198           21.246239
                   3           7052     18.462847           19.941130
                   4           6983     18.215667           21.561202


## 6. Conclusiones Generales y Diagnóstico de Modelos

### 1. Desconexión entre Rentabilidad y Retrasos Logísticos
El análisis empírico demuestra que el riesgo de retraso (`late_delivery_risk`) **no influye en la probabilidad de pérdida operativa**:
* La tasa de margen negativo se mantiene constante ($\approx 18.7\%$) en todos los rangos, desde adelantos de -2 días hasta retrasos de +4 días.
* La fórmula de beneficio financiero en la fuente de datos no contempla penalizaciones o sobrecostos derivados del tiempo de tránsito real.

### 2. Comparativa de Objetivos de Modelado

| Objetivo | Algoritmo | ROC-AUC | Viabilidad de Producción | Diagnóstico Técnico |
| :--- | :--- | :---: | :---: | :--- |
| **Operación con Pérdida** | Regresión Logística | 0.5257 |  No Viable | Falta de variables de costos reales/directos en el dataset. |
| **Operación con Pérdida** | XGBoost (V1 / V2) | 0.5161 |  No Viable | Comportamiento cercano al azar; ausencia de señal explicativa. |
| **Riesgo de Retraso** | Clasificador Logístico/XGB | **0.7812** | ** Viable** | Fuerte relación entre modalidad de envío, regiones y tiempos. |

### 3. Recomendaciones Ejecutivas
1. **Redirección del Caso de Uso:** Implementar el modelo enfocado en la prevención de **`late_delivery_risk`** ($\text{ROC-AUC} = 0.7812$), ya que permite a la cadena de suministro intervenir logísticamente pedidos antes de que expire el tiempo prometido.
2. **Requisitos para el Modelado Financiero:** Si la organización requiere predecir márgenes negativos, se debe auditar la captura de datos e incorporar tablas de costos logísticos variables, tasas de devolución por categoría y comisiones por pasarela de pago.